In [1]:
import pandas as pd
import numpy as np
import os

from sklearn.metrics import average_precision_score
import xgboost as xgb

In [9]:
CONFIG = {
    "n_estimators": 500,
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42
}

In [10]:
TARGET = "deterioration_next_12h"

df = pd.read_csv("../data/features/features.csv")

df.head()

,hour_from_admission,heart_rate,respiratory_rate,spo2_pct,temperature_c,systolic_bp,diastolic_bp,oxygen_device,oxygen_flow,mobility_score,...,lactate_roll_mean,lactate_roll_std,lactate_cv,tachycardia_flag,temp_diff,fever_trend,hour_sin,hour_cos,time_bucket,sofa_proxy
0,0,68.58,14.47,96.52,37.18,108.94,78.43,4,0.0,2,...,1.183333,0.063456,0.053625,0,0.07,0,0.000000,1.000000,0,-2.91
1,1,67.03,13.87,94.94,37.25,111.73,79.14,4,0.0,3,...,1.183333,0.063456,0.053625,0,0.07,0,0.258819,0.965926,0,-4.27
2,2,69.05,14.63,94.45,37.29,111.48,78.86,4,0.0,2,...,1.183333,0.063456,0.053625,0,0.04,0,0.500000,0.866025,0,-3.47
3,3,69.07,14.42,95.16,37.27,110.68,76.79,4,0.0,2,...,1.183333,0.063456,0.053625,0,-0.02,0,0.707107,0.707107,0,-3.47
4,4,73.35,15.62,95.83,37.21,110.38,75.47,4,0.0,3,...,1.183333,0.063456,0.053625,0,-0.06,0,0.866025,0.500000,0,-3.80


In [11]:
def clean_dataframe(df):
    df = df.copy()
    
    # Convert object → numeric safely
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].replace(["none", "None", "nan"], np.nan)
            df[col] = df[col].astype("category").cat.codes
    
    # Force numeric
    df = df.apply(pd.to_numeric, errors="coerce")
    
    # Fill missing
    df = df.fillna(0)
    
    return df

In [12]:
EXCLUDE = ["patient_id", TARGET]

FEATURES = [
    col for col in df.columns
    if col not in EXCLUDE
]

print("Total features:", len(FEATURES))

Total features: 58


In [13]:
def train_xgb(train_df, val_df, fold):
    
    # Clean data
    train_df = clean_dataframe(train_df)
    val_df = clean_dataframe(val_df)
    
    X_train = train_df[FEATURES]
    y_train = train_df[TARGET].astype(int)
    
    X_val = val_df[FEATURES]
    y_val = val_df[TARGET].astype(int)
    
    # Handle class imbalance safely
    pos = (y_train == 1).sum()
    neg = (y_train == 0).sum()
    
    scale_pos_weight = neg / pos if pos > 0 else 1
    
    model = xgb.XGBClassifier(
        n_estimators=CONFIG["n_estimators"],
        learning_rate=CONFIG["learning_rate"],
        max_depth=CONFIG["max_depth"],
        subsample=CONFIG["subsample"],
        colsample_bytree=CONFIG["colsample_bytree"],
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        tree_method="hist",
        random_state=CONFIG["random_state"],
        n_jobs=-1
    )
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    
    preds = model.predict_proba(X_val)[:, 1]
    
    pr_auc = average_precision_score(y_val, preds)
    
    print(f"Fold {fold} PR-AUC: {pr_auc:.4f}")
    
    # Save model
    model.save_model(f"../models/xgboost/xgb_fold_{fold}.json")
    
    return pr_auc

In [14]:
os.makedirs("../models/xgboost", exist_ok=True)
os.makedirs("../outputs/metrics", exist_ok=True)

scores = []

for fold in range(5):
    print(f"\n==== Fold {fold} ====")
    
    train_df = pd.read_csv(f"../data/splits/train_fold_{fold}.csv")
    val_df = pd.read_csv(f"../data/splits/val_fold_{fold}.csv")
    
    score = train_xgb(train_df, val_df, fold)
    scores.append(score)


==== Fold 0 ====
Fold 0 PR-AUC: 0.6718

==== Fold 1 ====
Fold 1 PR-AUC: 0.6716

==== Fold 2 ====
Fold 2 PR-AUC: 0.6355

==== Fold 3 ====
Fold 3 PR-AUC: 0.6435

==== Fold 4 ====
Fold 4 PR-AUC: 0.6964


In [15]:
scores = np.array(scores)

print("\nXGBoost Results:")
print("Mean PR-AUC:", scores.mean())
print("Std:", scores.std())

pd.DataFrame({"pr_auc": scores}).to_csv("../outputs/metrics/xgb_cv.csv", index=False)


XGBoost Results:
Mean PR-AUC: 0.663755652397464
Std: 0.021915890781077217
